# Lecture 1: Simple Linear Regression - Weight and Height

### Short, simple, self-study notes

Simple Linear Regression predicts one numeric value using one input feature. In this example, we predict a person's height from weight.

**Main idea:** Find the best straight line that keeps the overall prediction errors as small as possible.

## 1. The problem

- **Independent feature X:** Weight
- **Dependent target y:** Height

The model learns a line:

**predicted height = intercept + coefficient × weight**

The intercept is where the line starts. The coefficient is the slope: how much the predicted height changes when weight changes by one unit.

This small dataset is only for learning. A real height model needs more data and should consider many other factors.

## 2. Load the dataset and view the relationship

A scatter plot is the best first visual for simple linear regression. If the points roughly follow an upward or downward line, a straight-line model may be useful.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_path = Path("Simple Linear Regression Practicals") / "height-weight.csv"
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["Weight"], df["Height"], color="#457b9d", s=55)
plt.title("Weight and height data points", weight="bold")
plt.xlabel("Weight")
plt.ylabel("Height")
plt.grid(alpha=0.25)
plt.show()

## 3. Train-test split

We divide the data before fitting the model:

- **Training data:** used to learn the line.
- **Test data:** kept aside to check predictions on unseen rows.

Here, 80% of rows are used for training and 20% for testing. Random state makes the same split happen each time.

In [ ]:
X = df[["Weight"]]  # Double brackets keep X as a two-dimensional table.
y = df["Height"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 4. Standardization without data leakage

Standardization changes a value using the training mean and training standard deviation:

**z = (value - training mean) / training standard deviation**

For simple LinearRegression in scikit-learn, scaling is not required to get a valid fit. It is shown here because it is essential for many optimisation-based and regularized models, and it teaches the safe pattern:

- Use fit_transform on X_train.
- Use transform on X_test.
- Never fit the scaler on test data.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

regressor = LinearRegression()
regressor.fit(X_train_scaled, y_train)

y_pred_train = regressor.predict(X_train_scaled)
y_pred_test = regressor.predict(X_test_scaled)

print("Intercept:", round(regressor.intercept_, 3))
print("Coefficient for scaled weight:", round(regressor.coef_[0], 3))

## 5. Visual: best-fit line

The blue points are training examples. The red line is the height predicted by the fitted model for each weight.

The line is not required to pass through every point. It is chosen to make the overall squared prediction errors as small as possible.

In [ ]:
weight_line = np.linspace(X_train["Weight"].min(), X_train["Weight"].max(), 100).reshape(-1, 1)
height_line = regressor.predict(scaler.transform(weight_line))

plt.figure(figsize=(7, 4))
plt.scatter(X_train["Weight"], y_train, label="training points", color="#457b9d", s=55)
plt.plot(weight_line, height_line, label="best-fit line", color="#e76f51", linewidth=2.5)
plt.xlabel("Weight")
plt.ylabel("Height")
plt.title("Simple Linear Regression best-fit line", weight="bold")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Evaluate the unseen test predictions

- **MSE:** average squared error; larger mistakes are penalized more.
- **MAE:** average absolute error in height units; easier to explain.
- **RMSE:** square root of MSE; also in height units.
- **R²:** fraction of target variation explained by the model.
- **Adjusted R²:** adjusts R² for the number of predictors. With one predictor, it will be close to R².

For MSE, MAE, and RMSE, smaller is better. For R², higher is generally better on test data.

In [ ]:
mse = mean_squared_error(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_test)

n_test = len(y_test)
number_of_predictors = X_test.shape[1]
adjusted_r2 = 1 - (1 - r2) * (n_test - 1) / (n_test - number_of_predictors - 1)

metrics = pd.Series({
    "MSE": mse,
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "Adjusted R2": adjusted_r2,
}).round(3)
metrics

## 7. Prediction plot and residual checks

A residual is:

**residual = actual height - predicted height**

For a good linear model, residuals should be spread around zero without a strong curved pattern. With only five test rows, these plots are educational hints, not strong proof.

In [ ]:
residuals = y_test - y_pred_test
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(y_test, y_pred_test, color="#457b9d", s=55)
low = min(y_test.min(), y_pred_test.min())
high = max(y_test.max(), y_pred_test.max())
axes[0].plot([low, high], [low, high], "--", color="#e76f51")
axes[0].set_xlabel("Actual height")
axes[0].set_ylabel("Predicted height")
axes[0].set_title("Actual versus predicted")

axes[1].scatter(y_pred_test, residuals, color="#457b9d", s=55)
axes[1].axhline(0, linestyle="--", color="#e76f51")
axes[1].set_xlabel("Predicted height")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals should scatter around zero")

fig.suptitle("Simple regression diagnostic diagrams", weight="bold")
fig.tight_layout()
plt.show()

## 8. Predict height for a new weight

A new input must go through the same scaler used during training. Give the model a two-dimensional table, even for one value.

In [ ]:
new_weight = pd.DataFrame({"Weight": [80]})
new_weight_scaled = scaler.transform(new_weight)
predicted_height = regressor.predict(new_weight_scaled)[0]

print(f"Predicted height for weight 80: {predicted_height:.2f}")

## 9. Final revision card

- Simple Linear Regression uses one input feature to predict one numeric target.
- Split data into training and test sets before fitting.
- Keep X as a two-dimensional table, even when there is only one feature.
- Fit scaling on training data and only transform test or new data.
- The best-fit line minimizes overall squared errors.
- Use MAE, MSE, RMSE, R², and adjusted R² to evaluate predictions.
- Residuals are actual minus predicted values; inspect whether they are spread around zero.
- Use the same trained scaler and model to predict a new data point.

### One-line interview answer

**Simple Linear Regression learns a best-fit straight line between one feature and one numeric target, then we evaluate its unseen predictions using error metrics and R².**